In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

import matplotlib.pyplot as plt

In [2]:
transform = transforms.ToTensor()

train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

In [3]:
from turtle import forward


class Generator(nn.Module):
    def __init__(self, noise_dim=100):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(noise_dim, 128),
            nn.ReLU(),

            nn.Linear(128, 256),
            nn.ReLU(),

            nn.Linear(256, 512),
            nn.ReLU(),
            
            nn.Linear(512, 784),
            nn.Sigmoid()
        )

    def forward(self, z):
        return self.net(z)

In [4]:
noise_dim = 100

generator = Generator(noise_dim)

z = torch.randn(4, noise_dim)

fake_images = generator(z)

print("Noise shape:", z.shape)
print("Generator shape:", fake_images.shape)

Noise shape: torch.Size([4, 100])
Generator shape: torch.Size([4, 784])


In [5]:
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(784, 512),
            nn.LeakyReLU(0.2),

            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),

            nn.Linear(256, 1)
        )

    def forward(self, x):
        return self.net(x)

In [6]:
discriminator = Discriminator()

scores = discriminator(fake_images)

print("Input shape:", fake_images.shape)
print("Discriminator output shape:", scores.shape)
print(scores)

Input shape: torch.Size([4, 784])
Discriminator output shape: torch.Size([4, 1])
tensor([[0.0582],
        [0.0581],
        [0.0578],
        [0.0586]], grad_fn=<AddmmBackward0>)


In [7]:
noise_dim = 100

generator = Generator(noise_dim)
discriminator = Discriminator()

criterion = nn.BCEWithLogitsLoss()

g_optimizer = optim.Adam(
    generator.parameters(),
    lr=0.0002
)

d_optimizer = optim.Adam(
    discriminator.parameters(),
    lr=0.0002
)

In [8]:
print("Generator parameters:",
      sum(p.numel() for p in generator.parameters()))

print("Discriminator parameters:",
      sum(p.numel() for p in discriminator.parameters()))

Generator parameters: 579728
Discriminator parameters: 533505


In [9]:
# Get a batch of real images
real_images, _ = next(iter(train_loader))

# Flatten images: [64, 1, 28, 28] -> [64, 784]
real_images = real_images.view(-1, 784)

batch_size = real_images.size(0)

# Labels for real images
real_labels = torch.ones(batch_size, 1)

# Train Discriminator on real images
real_outputs = discriminator(real_images)
real_loss = criterion(real_outputs, real_labels)

# Generate fake images
z = torch.randn(batch_size, noise_dim)
fake_images = generator(z)

# Labels for fake images
fake_labels = torch.zeros(batch_size, 1)

# Train Discriminator on fake images
fake_outputs = discriminator(fake_images.detach())
fake_loss = criterion(fake_outputs, fake_labels)

# Combine losses
d_loss = real_loss + fake_loss

# Update Discriminator
d_optimizer.zero_grad()
d_loss.backward()
d_optimizer.step()

print("Real loss:", real_loss.item())
print("Fake loss:", fake_loss.item())
print("Total Discriminator loss:", d_loss.item())

Real loss: 0.7097219228744507
Fake loss: 0.6665592789649963
Total Discriminator loss: 1.3762812614440918


c:\Users\boost\anaconda3\Lib\site-packages\torch\autograd\graph.py:882: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 11060). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\c10\cuda\CUDAFunctions.cpp:119.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


In [11]:
# Generate new fake images
z = torch.randn(batch_size, noise_dim)
fake_images = generator(z)

# Pass fake images through Discriminator
outputs = discriminator(fake_images)

# Generator wants Discriminator to classify them as real
labels = torch.ones_like(outputs)
g_loss = criterion(outputs, labels)

# Update Generator
g_optimizer.zero_grad()
g_loss.backward()
g_optimizer.step()

print("Generator loss:", g_loss.item())

Generator loss: 0.8504033088684082


In [12]:
generator = Generator(noise_dim)
discriminator = Discriminator()

g_optimizer = optim.Adam(
    generator.parameters(),
    lr=0.0002
)

d_optimizer = optim.Adam(
    discriminator.parameters(),
    lr=0.0002
)

In [13]:
from numpy import real


num_epochs = 10

for epoch in range(num_epochs):
    for real_images, _ in train_loader:

        # Prepare real images
        real_images =  real_images.view(-1, 784)
        batch_size =  real_images.size(0)

        # =====================
        # 1. Train Discriminator
        # =====================

        real_labels = torch.ones(batch_size, 1)
        fake_labels = torch.zeros(batch_size, 1)

        # Real images
        real_outputs = discriminator(real_images)
        real_loss = criterion(real_outputs, real_labels)

        # Fake images
        z = torch.randn(batch_size, noise_dim)
        fake_images = generator(z)

        fake_outputs = discriminator(fake_images.detach())
        fake_loss = criterion(fake_outputs, fake_labels)

        # Total Discriminator Loss
        d_loss = real_loss + fake_loss

        d_optimizer.zero_grad()
        d_loss.backward()
        d_optimizer.step()

        # =========================
        # 2. Train Generator
        # =========================

        z = torch.randn(batch_size, noise_dim)
        fake_images = generator(z)

        outputs = discriminator(fake_images)
        g_loss = criterion(outputs, real_labels)

        g_optimizer.zero_grad()
        g_loss.backward()
        g_optimizer.step()

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"D Loss: {d_loss.item():.4f} "
        f"G Loss: {g_loss.item():.4f}"
    )

Epoch [1/10] D Loss: 0.0168 G Loss: 6.9697
Epoch [2/10] D Loss: 0.0108 G Loss: 5.9983
Epoch [3/10] D Loss: 0.0024 G Loss: 9.3549
Epoch [4/10] D Loss: 0.0070 G Loss: 7.4974
Epoch [5/10] D Loss: 0.0026 G Loss: 15.2105
Epoch [6/10] D Loss: 0.0362 G Loss: 7.1142
Epoch [7/10] D Loss: 0.0032 G Loss: 8.3064
Epoch [8/10] D Loss: 0.0002 G Loss: 9.7756
Epoch [9/10] D Loss: 0.0008 G Loss: 10.9226
Epoch [10/10] D Loss: 0.0006 G Loss: 8.0195
